# Install & Import

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session


session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
STAGE = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"

In [ ]:
TARGET_COLS = [
    'Electrical Conductance',
    'Total Alkalinity',
    'Dissolved Reactive Phosphorus'
]
KEY_COLS_FULL = ['Latitude', 'Longitude', 'Sample_Date', 'Sample Date',
                 'GEIMS Station Number', 'River Name']

SANLC_IMPACT_MAP = {
    'mines: extraction pits, quarries': 4,
    'commercial': 3,
    'commercial annual crops pivot irrigated': 3,
    'commercial annual crops rain-fed / dryland': 3,
    'cultivated commercial permanent orchards': 3,
    'cultivated commercial permanent vines': 3,
    'cultivated commercial sugarcane non-pivot': 3,
    'subsistence / small-scale annual crops': 2,
    'residential formal (low veg / grass)': 2,
    'smallholdings (low veg / grass)': 2,
    'fallow land & old fields (grass)': 1,
    'fallow land & old fields (trees)': 1,
    'other bare': 1,
    'natural grassland': 0,
    'dense forest & woodland': 0,
    'open woodland': 0,
    'contiguous (indigenous) forest': 0,
    'contiguous & dense plantation forest': 0,
    'contiguous low forest & thicket': 0,
    'low shrubland (fynbos)': 0,
    'low shrubland (nama karoo)': 0,
    'low shrubland (succulent karoo)': 0,
    'herbaceous wetlands (previously mapped)': 0,
    'Unclassified/No Data': -1,
}

def load_from_stage(filename):
    """Download file dari Stage ke /tmp/ lalu load. O(1) I/O jika sudah ada."""
    local_path = f"/tmp/{filename}"
    if not os.path.exists(local_path):
        session.file.get(f"{STAGE}/{filename}", "/tmp/")
    if filename.endswith('.parquet'):
        return pd.read_parquet(local_path)
    return pd.read_csv(local_path)

def get_feature_cols(df):
    """Return hanya kolom fitur (bukan key/target). O(n_cols)."""
    exclude = set(KEY_COLS_FULL + TARGET_COLS)
    return [c for c in df.columns if c not in exclude]

def encode_sanlc_column(df, col):
    """Encode 1 kolom SANLC text → numeric impact score. O(n_rows)."""
    new_col = col.replace('_class_', '_impact_')
    df[new_col] = df[col].map(SANLC_IMPACT_MAP).fillna(-1).astype(np.int8)
    return df.drop(columns=[col])

def upload_to_stage(df, filename):
    """Save parquet & upload ke Stage. O(n_rows)."""
    path = f"/tmp/{filename}"
    df.to_parquet(path, index=False, engine='pyarrow', compression='snappy')
    session.file.put(f"file://{path}", STAGE, auto_compress=False, overwrite=True)
    mb = os.path.getsize(path) / 1024 / 1024
    print(f"✓ {filename}: {df.shape} → {mb:.2f} MB")

print("Setup done")

# Data Ingestion

In [ ]:
target_train = load_from_stage("water_quality_training_dataset.csv")
target_test = load_from_stage("submission_template.csv")

for df in [target_train, target_test]:
    df['Sample_Date'] = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=False)

data_map = {
    'landsat_train': 'landsat_features_training.csv',
    'landsat_test':  'landsat_features_validation.csv',
    'terra_train':   'terraclimate_features_training.csv',
    'terra_test':    'terraclimate_features_validation.csv',
}
ey_data = {k: load_from_stage(v) for k, v in data_map.items()}

ext_files = {
    'elevation':  ('elevation_features.parquet',       'test_elevation_features.parquet'),
    'weather':    ('weather_features.parquet',         'test_weather_features.parquet'),
    'soilgrids':  ('soilgrids_features.parquet',       'test_soilgrids_features.parquet'),
    'sanlc':      ('train_sanlc_features.parquet',     'test_sanlc_features.parquet'),
    'worldpop':   ('train_worldpop_features.parquet',  'test_worldpop_features.parquet'),
    'hydro':      ('train_hydro_features.parquet',     'test_hydro_features.parquet'),
}
ext_train = {}
ext_test = {}
for name, (f_train, f_test) in ext_files.items():
    try:
        ext_train[name] = load_from_stage(f_train)
        ext_test[name] = load_from_stage(f_test)
        print(f" {name}: train={ext_train[name].shape}, test={ext_test[name].shape}")
    except Exception as e:
        print(f" {name}: SKIP — {e}")

print(f"\n Target train: {target_train.shape}, test: {target_test.shape}")

## Encode

In [ ]:
sanlc_text_cols = ['sanlc2022_class_1km', 'sanlc2020_class_1km']

for split_name, split_dict in [('train', ext_train), ('test', ext_test)]:
    if 'sanlc' in split_dict:
        df = split_dict['sanlc']
        for col in sanlc_text_cols:
            if col in df.columns:
                df = encode_sanlc_column(df, col)
        split_dict['sanlc'] = df
        print(f"  ✓ SANLC {split_name} encoded: {df.columns.tolist()}")

# Juga buat SANLC change detection: apakah lahan berubah antara 2020-2022?
for split_dict in [ext_train, ext_test]:
    if 'sanlc' in split_dict:
        df = split_dict['sanlc']
        if 'sanlc2022_impact_1km' in df.columns and 'sanlc2020_impact_1km' in df.columns:
            df['sanlc_change_2020_2022'] = (
                df['sanlc2022_impact_1km'] - df['sanlc2020_impact_1km']
            ).astype(np.int8)
        split_dict['sanlc'] = df

print("SANLC encoding selesai")

## Merging

In [ ]:
def master_merge(target_df, ey_dict, ext_dict, split='train'):
    """
    Merge semua dataset ke target. 
    - Static features: merge on [Latitude, Longitude]
    - Time-dependent: merge on [Latitude, Longitude, Sample_Date] atau concat by index
    O(n_datasets × n_rows × log(n_keys)) via pandas merge (hash join internally)
    """
    df = target_df.copy()
    
    for ey_name in ['landsat', 'terra']:
        ey_key = f'{ey_name}_{split}'
        if ey_key not in ey_dict:
            continue
        ey_df = ey_dict[ey_key]
        feat_cols = get_feature_cols(ey_df)
        if not feat_cols:
            continue
        
    
        has_keys = 'Latitude' in ey_df.columns and 'Longitude' in ey_df.columns
        if has_keys and len(ey_df) == len(df):
        
            if 'Sample Date' in ey_df.columns and 'Sample_Date' not in ey_df.columns:
                ey_df['Sample_Date'] = pd.to_datetime(ey_df['Sample Date'], format='mixed', dayfirst=False)
            
            merge_keys = ['Latitude', 'Longitude']
            if 'Sample_Date' in ey_df.columns:
                merge_keys.append('Sample_Date')
            
            df = df.merge(
                ey_df[merge_keys + feat_cols].drop_duplicates(subset=merge_keys),
                on=merge_keys, how='left', suffixes=('', f'_{ey_name}')
            )
        else:
            df = pd.concat([df.reset_index(drop=True),
                           ey_df[feat_cols].reset_index(drop=True)], axis=1)
        print(f"  + {ey_name}: +{len(feat_cols)} features → {df.shape}")
    
    # External features
    for name, ext_df in ext_dict.items():
        feat_cols = get_feature_cols(ext_df)
        if not feat_cols:
            continue
        
        # Tentukan merge keys
        has_date = 'Sample_Date' in ext_df.columns
        merge_keys = [c for c in ['Latitude', 'Longitude'] if c in ext_df.columns]
        if has_date:
            merge_keys.append('Sample_Date')
        
        if merge_keys:
            merge_df = ext_df[merge_keys + feat_cols].drop_duplicates(subset=merge_keys)
            df = df.merge(merge_df, on=merge_keys, how='left', suffixes=('', f'_{name}'))
        else:
            df = pd.concat([df.reset_index(drop=True),
                           ext_df[feat_cols].reset_index(drop=True)], axis=1)
        print(f"  + {name}: +{len(feat_cols)} features → {df.shape}")
    
    return df

print("=" * 60)
print("MASTER MERGE — Training Data")
print("=" * 60)
master_train = master_merge(target_train, ey_data, ext_train, 'train')

print(f"\n{'=' * 60}")
print("MASTER MERGE — Test Data")
print("=" * 60)
master_test = master_merge(target_test, ey_data, ext_test, 'test')

print(f"\n FINAL — master_train: {master_train.shape}")
print(f"FINAL — master_test:  {master_test.shape}")

# Sanity Check's

In [ ]:
numeric_cols = master_train.select_dtypes(include=[np.number]).columns
master_train[numeric_cols] = master_train[numeric_cols].fillna(0)

test_numeric = numeric_cols.intersection(master_test.columns)
master_test[test_numeric] = master_test[test_numeric].fillna(0)

# Missing values report
missing = master_train.isnull().sum()
has_missing = missing[missing > 0]
if len(has_missing) > 0:
    print("Kolom masih ada NaN (non-numeric, perlu dicek):")
    print(has_missing)
else:
    print(" Semua kolom bersih — 0 missing values")

# Duplikat check
n_dup = master_train.duplicated().sum()
print(f"\n  Duplicate rows: {n_dup}")
print(f"  Train shape: {master_train.shape}")
print(f"  Test shape:  {master_test.shape}")
feat_count = len([c for c in master_train.columns if c not in set(KEY_COLS_FULL + TARGET_COLS)])
print(f"  Total feature columns: {feat_count}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']
transform_recommendations = {}

for i, col in enumerate(TARGET_COLS):
    data = master_train[col].dropna()
    skew = data.skew()
    
    axes[i].hist(data, bins=50, edgecolor='black', alpha=0.7, color=colors[i])
    axes[i].set_title(f'{col.split("(")[0].strip()}\nskew={skew:.2f}, median={data.median():.1f}', fontsize=10)
    axes[i].axvline(data.median(), color='red', linestyle='--', label=f'Median')
    axes[i].legend(fontsize=8)
    
    transform_recommendations[col] = 'log1p' if abs(skew) > 1 else 'none'
    symbol = 'PERLU log-transform' if abs(skew) > 1 else '✓ OK'
    print(f"{col}: skew={skew:.2f} → {symbol}")

plt.suptitle('Distribution of 3 Target Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Simpan rekomendasi untuk notebook 05
print(f"\nTransform recommendations: {transform_recommendations}")

In [ ]:
print("Correlation to target:")
target_corr = master_train[TARGET_COLS].corr()
print(target_corr.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(target_corr, annot=True, cmap='coolwarm', center=0, ax=axes[0], fmt='.3f')
axes[0].set_title('Inter-Target Correlation')

all_numeric = master_train.select_dtypes(include=[np.number])
corr_matrix = all_numeric.corr()

top_features_per_target = {}
for target in TARGET_COLS:
    if target not in corr_matrix.columns:
        continue
    target_corrs = corr_matrix[target].drop(TARGET_COLS, errors='ignore').abs().sort_values(ascending=False).head(10)
    top_features_per_target[target] = target_corrs.index.tolist()
    short_name = target.split('(')[0].strip()
    print(f"\nTop 10 features for {short_name}:")
    for feat in target_corrs.index:
        print(f"  {feat}: {corr_matrix[target][feat]:.4f}")

# Heatmap top features gabungan
all_top = list(set(f for feats in top_features_per_target.values() for f in feats))
if all_top:
    sub_corr = master_train[all_top + TARGET_COLS].corr()[TARGET_COLS].drop(TARGET_COLS, errors='ignore')
    sns.heatmap(sub_corr, annot=True, cmap='RdBu_r', center=0, ax=axes[1], fmt='.2f', 
                yticklabels=[c[:25] for c in sub_corr.index])
    axes[1].set_title('Top Features vs Targets')

plt.tight_layout()
plt.show()

# Extract to Parquet

In [ ]:
drop_cols = ['GEIMS Station Number', 'River Name', 'Sample Date', 'Sample_Date',
             'Latitude', 'Longitude']

train_export = master_train.drop(columns=[c for c in drop_cols if c in master_train.columns])
test_export = master_test.drop(columns=[c for c in drop_cols if c in master_test.columns])

upload_to_stage(train_export, "master_train.parquet")
upload_to_stage(test_export, "master_test.parquet")

meta = {
    'target_cols': TARGET_COLS,
    'n_features': len([c for c in train_export.columns if c not in TARGET_COLS]),
    'n_train_rows': len(train_export),
    'n_test_rows': len(test_export),
    'transform_recommendations': transform_recommendations,
    'feature_names': [c for c in train_export.columns if c not in TARGET_COLS]
}
with open('/tmp/pipeline_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
session.file.put("file:///tmp/pipeline_meta.json", STAGE, auto_compress=False, overwrite=True)

print(f"\n✓ Exported: {meta['n_features']} features, {meta['n_train_rows']} train rows")
print(f"  Feature list: {meta['feature_names'][:10]}... (total {meta['n_features']})")